# SECOM – Week 2: Data preparation

Armand Bussolino, Aurélien Roche, Jérémie Leroy-Dubief

Goal: understand the missing values and the redundancy of the features, and decide which cleaning steps to test.
All analyses use the **training set only**, so the test set does not influence any preprocessing choice.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # run from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.secom import load_secom, get_split
from scipy.stats import fisher_exact, mannwhitneyu
from scipy.sparse.csgraph import connected_components
from src.secom import DropCorrelated

X, y = load_secom()
X_train, X_test, y_train, y_test = get_split(X, y)
print('Train:', X_train.shape, '| Fail:', int(y_train.sum()))

Train: (1253, 590) | Fail: 83


## 1. Missing values and the target

Do failing samples have more missing values than passing ones?

In [2]:
row_missing = X_train.isna().mean(axis=1) * 100
print(row_missing.groupby(y_train.map({0: 'Pass', 1: 'Fail'})).describe().round(2))
print('Mann-Whitney p-value:', round(mannwhitneyu(row_missing[y_train == 1], row_missing[y_train == 0]).pvalue, 3))

       count  mean   std   min   25%   50%   75%    max
fail                                                   
Fail    83.0  4.22  1.54  1.36  3.39  4.07  4.75   8.14
Pass  1170.0  4.61  2.38  0.68  3.39  4.07  5.42  25.76
Mann-Whitney p-value: 0.163


Is the fact that a given sensor is missing linked to failure? Fisher exact test per feature, with a Benjamini-Hochberg correction because we run hundreds of tests.

In [3]:
rows = []
for col in X_train.columns:
    miss = X_train[col].isna()
    if 0 < miss.sum() < len(miss):
        table = pd.crosstab(miss, y_train).reindex(index=[False, True], columns=[0, 1], fill_value=0)
        rows.append({'feature': col, 'missing (%)': 100 * miss.mean(),
                     'fail rate if missing (%)': 100 * y_train[miss].mean(),
                     'fail rate if present (%)': 100 * y_train[~miss].mean(),
                     'p': fisher_exact(table)[1]})
miss_test = pd.DataFrame(rows).sort_values('p')

# Benjamini-Hochberg adjusted p-values
m = len(miss_test)
q = miss_test['p'].to_numpy() * m / np.arange(1, m + 1)
miss_test['q (BH)'] = np.minimum.accumulate(q[::-1])[::-1].clip(max=1)

print('Features tested:', m)
print('p < 0.05 :', int((miss_test['p'] < 0.05).sum()))
print('q < 0.05 :', int((miss_test['q (BH)'] < 0.05).sum()))
miss_test.head(10).round(3)

Features tested: 506
p < 0.05 : 4
q < 0.05 : 0


,feature,missing (%),fail rate if missing (%),fail rate if present (%),p,q (BH)
63,sensor_72,49.082,4.715,8.464,0.009,1.0
64,sensor_73,49.082,4.715,8.464,0.009,1.0
299,sensor_346,49.082,4.715,8.464,0.009,1.0
298,sensor_345,49.082,4.715,8.464,0.009,1.0
216,sensor_247,45.331,5.282,7.737,0.088,1.0
100,sensor_112,45.331,5.282,7.737,0.088,1.0
450,sensor_519,45.331,5.282,7.737,0.088,1.0
335,sensor_385,45.331,5.282,7.737,0.088,1.0
401,sensor_467,0.239,33.333,6.560,0.186,1.0
397,sensor_463,0.239,33.333,6.560,0.186,1.0


Missing values come in blocks: many features are missing on exactly the same samples.

In [4]:
masks = X_train.isna()
masks = masks.loc[:, masks.any()]
patterns = masks.T.drop_duplicates()
print('Features with missing values:', masks.shape[1])
print('Distinct missing patterns   :', len(patterns))

Features with missing values: 506
Distinct missing patterns   : 34


**Observations.** Failing samples do not have more missing values than passing ones. A few features show a link between missingness and failure, but none survives the multiple-testing correction: missingness does not carry information about the target. Missing values follow a few patterns only, so whole groups of sensors are missing together (probably steps not measured for some samples).

## 2. Redundant features

In [5]:
dc = DropCorrelated(threshold=0.98).fit(X_train)
print('Features kept after dropping |r| >= 0.98:', int(dc.keep_.sum()), '/', X_train.shape[1])

# Groups of features linked by |r| >= 0.98 (connected components of the correlation graph)
corr = X_train.corr(min_periods=100).abs().fillna(0).to_numpy(copy=True)
np.fill_diagonal(corr, 0)
n_comp, comp = connected_components(corr >= 0.98, directed=False)
sizes = pd.Series(comp).value_counts()
groups = sizes[sizes > 1]
print('Groups of redundant features:', len(groups), '| features in these groups:', int(groups.sum()))
print('Largest groups:', groups.head(5).tolist())

Features kept after dropping |r| >= 0.98: 446 / 590


Groups of redundant features: 94 | features in these groups: 241
Largest groups: [9, 6, 6, 4, 4]


**Observations.** About half of the non-constant features belong to groups of near-identical sensors. One feature per group is enough to keep the information.

## 3. Cleaning decisions

All steps are sklearn transformers in `src/secom.py`, fitted inside each training fold.

| Step | B1 (baseline) | Tested as a variant (notebook 03) | Reason |
|---|---|---|---|
| Median imputation | ✓ | | Robust to the skewed sensor values |
| Drop constant / empty features | ✓ | | 116 features carry no information |
| Standardization | ✓ | | Needed by logistic regression |
| Drop features with > 50% missing | | ✓ | Mostly imputed values, little real signal |
| Drop near-duplicates (\|r\| ≥ 0.98) | | ✓ | Removes duplicated information |
| Missing-value indicators | | ✓ | Checks whether missingness is predictive |

We do not remove outliers: the sensors are anonymized, so we cannot tell a measurement error from a real process deviation, and failures may be exactly those deviations.